# Week 8: PCA, SVD, explained variance, and dimensional reduction

In last week's notebook you learned how to use the SVD and sci-kit learn to perform very basis principal component analysis. In this follow-up notebook, we'll cover some important topics in a little more depth.

1. The transpose exchanges $H$ and $CT$; using $CT$ instead of $H$ for PCA
1. The SVD, the covariance matrix, singular values, and explained variance
1. The SVD and approximating a matrix with a lower rank approximation



## Initialization cells

The imports are a little different this time: we're asking you to import some additional standard data packages: seaborn which provides data visualization using matplotlib, pandas that provides a lot of dataset tools built on numpy, and scikit-learn or sklearn that provides lots of machine learning tools and examples.

In [ ]:
import numpy as np
import numpy.linalg as la
import matplotlib.pyplot as plt

from sklearn import datasets
from sklearn.decomposition import PCA


## End of initiatlization cells

## The SVD and the transpose

An important simple fact about the transpose is that it reverses order of multiplication:

>**Fact:** If $X$ is an $a\times b$ matrix and $Y$ is a $b\times c$ matrix, so that $XY$ is an $a\times c$ matrix, then 
$$
      (XY)^T = Y^T X^T
$$
>or in Python
````
   (X@Y).T = X.T @ Y.T
````

This simple fact means that the SVD of $X$ and the SVD of $X.T$ are related to each other in a very simply way:

>**Fact:** If $X$ is an $a\times b$ matrix with SVD
````
     X = (H*s) @ C.T
````
>then the SVD of $X^T$ is
````
    X = (C*s) @ H.T
````


This means that in our SVD approach to PCA, we can save a step. Recall the steps from last time: 

1. Start with an $n \times k$ matrix of data $X$. You have $n$ observations of a $k$-dimenionsal vector.  
2. Center the data by subtracting the mean of each column: you get a new matrix $Xcentered$.

````
Xmean = np.mean(X,axis=0)
Xcentered = X - Xmean
````

3. Transpose the centered data to form a matrix $M = Xcentered.T$
4. Now $M$ is a $k\times n$ matrix that takes each $n$-dimensional standard basis vector $e_j$ to one of your centered observations.
5. Perform the SVD (and you can set `full_matrices=False`): 
````
H,s,CT = la.svd(M,full_matrices=False)
````
Then each columns of $H$ is a principal component of your centered data. They are ranked in order of importance: $H[:,:0]$ is most important, $H[:,:0]$ is next.

Last week you did a lot of things with the singular values $s$ and the matrix $H$. What you've just seen about the SVD of the tranpose tells you **there is no need to make the matrix $M$**. Instead just perform the SVD on $Xcentered$:
````
U,sX,VT = la.svd(Xcentered,full_matrices=False)
````
Then
1. the singular values $s$ and $sX$ are the same
2. The matrix $H$ you used so much last week is the same as $VT.T$, up to changing the sign of each column (remember this ambiguity of the SVD)

So there is no need to transpose: just compute the SVD of your centered data matrix. 


Let's see this in our faces example from last week.

Once again we'll import some faces from scikit-learn's faces data set.

In [ ]:
# The min_faces_per_person tells sklearn to provide only the images of those people 
# who appear 60 or more times in the data base
faces = datasets.fetch_lfw_people(min_faces_per_person=60)

## The SVD, the covariance matrix, PCA, and explained variance

As last week, we'll import from scikit-learn a data set of digital images. 

In [ ]:
# The min_faces_per_person tells sklearn to provide only the images of those people 
# who appear 60 or more times in the data base
faces = datasets.fetch_lfw_people(min_faces_per_person=60)

In [ ]:
# Form the centered data set. Store it in X this week, so we don't have to type Xcentered over and over again

faces_mean = np.mean(faces.data,axis=0)
X  = faces.data - faces_mean

print(f"X is a data set with {X.shape[0]} rows/observations.")
print(f"Each row is an observation of {X.shape[1]} variables, the pixels of a digital photograph.")

Last week you saw that the principle components of $X$ could be found by looking at the columns of the left matrix in the SVD of $X.T$. Let's see that that's the same as looking at the (transpose) of the right matrix in the SVD of $X.$ 

In [ ]:
M = X.T
H,s,CT = la.svd(M,full_matrices=False)

U,t,VT = la.svd(X,full_matrices=False)

HT = H.T


In [ ]:

print("The singular values are the same: ",np.allclose(s,t))

print("Matrices $H.T$ and $VT$ are the same: ", np.allclose(HT,VT))

The singular values should have matched above. But the matrices $H^T$ and $VT$ don't have to, because remember that the SVD has a sign ambiguity. What had better be true is that every row of $H^T$ differs from the corresponding row of $VT$ by no more than multiplicaiton by $+1$ or $-1$.

In [ ]:
 
#checks row-by-row whether $HT$ and $VT$ are the same up to a change of sign in each row

np.all(\
    [np.allclose(HT[row],VT[row]) or np.allclose(HT[row],-VT[row]) \
     for row in range(HT.shape[0])]\
        )

So, for example, we can see our "eigenfaces" just by looking at the rows of `VT`, and all the analysis involving singular values can use the singular values of the matrix `X`, *which are the same as the singular values of the matrix* `M=X.T`

In [ ]:
#fig, axes = plt.subplots(1, 6, figsize=(9, 4),
#                         subplot_kw={'xticks':[], 'yticks':[]},
#                         gridspec_kw=dict(hspace=0.1, wspace=0.1))

fig, ax = plt.subplots(1, 10, figsize=(20, 5),
    subplot_kw={'xticks':[], 'yticks':[]},
    gridspec_kw=dict(hspace=0.05, wspace=0.05))

for i in range(10):
    ax[i].imshow(VT[i].reshape(62, 47), cmap='bone')

## The SVD, the covariance matrix, singular values, and explained variance

Last week, we used scikit-learn's PCA class to analyze this faces data, and we saw a very useful plot of how much variance was explained by increasing the number of singular values (this is called a scree plot).

In [ ]:
pca = PCA(85)
transformed_data = pca.fit_transform(faces.data)

fig,ax=plt.subplots(1,1)
ax.plot(np.cumsum(pca.explained_variance_ratio_))
ax.set_xlabel('number of components')
ax.set_ylabel('cumulative explained variance');

Let's see where this information came from.


### Variance and the covariance matrix


If ${a_0,\ldots,a_{n}}$ are $n$ observations of something, say, temperature, then their *average* is 
$$
\bar{a} = \frac{a_0+\ldots+a_{n-1}}{n}.
$$
Once you know the average of your data  a nice next thing to know is how much the data varies from the average. One measure of that is the *variance*, which is
$$
\begin{align*}
\mathrm{Var}(a) & = \frac{1}{n}\left( (a_0-\bar{a})^2 + \ldots + (a_{n-1} - \bar{a})^2\right) \\
& = \frac{1}{n} (a - \bar{a})\cdot (a-\bar{a}),
\end{align*}
$$
or in Python
````
Var(a) = (1/n) * (a-np.mean(a)) @ (a-np.mean(a)).
````

You may have heard of the "standard deviation" which is just the square root of the variance,
$$
\sigma = \sqrt{\mathrm{Var}(a)}.
$$

The reason for squaring is to make every deviation from the mean, whether greater or less than the mean, contribute to the variance. If $a_0$ is less than $\bar{a}$ and $a_1$ is more than $\bar{a}$, we want both deviations to contribute to the variance, rather than canceling.

We've been centering our data: shifting our variables over by the mean. If our data has mean zero, then the variance is just

$$
\mathrm{Var}(a)= \frac{1}{n} (a_0^2 + \ldots + a_{n-1}^2).
$$
Notice that if we've put our data $a$ into a vector, then this is just
$$
\mathrm{Var}(a) =\frac{1}{n} a \cdot a
$$
or in Python 
````
Var(a) = (1/n) * a @ a
````
Notice that, up to scaling by the number of observations of data, the variance is just the dot product of the data with itself, or the square of the norm of the data: this is a new interpretation of the dot product.

Now let's return to the case of a centered data set $X$. We'll assume that there are $n$ rows, and each row is an observation of $k$ variables, such as our centered faces data. That is, we're assuming that the mean of each column of $X$ is zero.

>**Fact:** The matrix
$$
SCov(X) = \frac{1}{n-1} X^T X
$$
or in Python
````
SCov = 1/(n-1) * X.T @ X
````
is the **sample covariance matrix** of the data $X$. For now we won't worry about the switch from $n$ to $n-1$, or the word "sample".  The switch and word appear for important theoretical and philosophical reasons related to the fact that your digital photographs are only some of the possible digital photographs of these individuals one might encounter or try to model.

Let's make this matrix.



In [ ]:
X.shape

In [ ]:
# The number of rows is the 0 entry of the shape
nobs =  X.shape[0]

SCov = (1/(nobs-1)) * X.T @ X

# numpy's function cov also computes this matrix. rowvar=False says that each row
# is an observation and each column is a variable

np.allclose(SCov,np.cov(X,rowvar=False),atol=0.001)

>**Fact:** The diagonal entries of our sample covariance matrix are just the variances of our observed variables  (up to the switch from $n$ to $n-1$).

This is because if the columns of $X$ are
$$
X = 
\begin{bmatrix}
| & & | \\
v_0 & ... & v_{k-1}\\
| & & |
\end{bmatrix}
$$
then the $(j,j)$ entry of $X.T @ X$ is just $v_j @ v_j$. Since the mean of the column $v_j$ is 0, we saw above that the variance of the column $v_j$ is $(1/n) * v_j @ v_j$. 

What about the meaning of the off-diagonal entries?

>**Definition:** If $a$ and $b$ are vectors of $n$ observations of two different measuresments (think of temperature and relative humidity measured in the same place daily for a year), then their **covariance** is 
$$
\mathrm{Cov}(a,b) =  \frac{1}{n} (a - \bar{a}) \cdot (b-\bar{b}) 
$$
or in Python
````
Cov(a,b) = (1/n) * (a-np.mean(a)) @ (b-np.mean(b))
````
If $a$ and $b$ both have mean zero, then this simplifies to 
````
Cov(a,b) = (1/n) * a @ b.
````

This is a new idea: **when $a$ and $b$ have mean zero, the dot product $a@b$ measures how the entries of $a$ and $b$ tend to travel in the same direction, in opposite directions, or independently.** Look at 
$$
a @ b = a_0 * b_0 + \ldots + a_{n-1} * b_{n-1}.
$$

1. If $a$ is a large positive number when $b$ is a large positive number, then $a@b$ will be a larger positive number
1. If $a$ is a large positive number when $b$ is a large negative number, then $a@b$ will be a larged negative number
1. If $a$ is near zero when $b$ is large, then $a@b$ will be nearer to zero. More generally,
1. If $a$ and $b$ vary around $0$ independently, then $a@b$ will tend to be nearer to zero

Notice that if $a=b$ then the convariance of $a$ and $b$ is just the variance of $a$: so covariance is a generalization of variance.

>**Fact:** Up to scaling by the length, the dot product measures the covariance of vectors of data!











### The Covariance Matrix and the SVD

>**Fact:** If $X$ in $n\times k$ matrix with SVD
````
   X = H @ np.diag(s) @ C.T = (H*s) @ C.T
````
then the SVD of $X^TX$ and $XX^T$ are given by 
````
 X.T @ X = C @np.diag(s**2) @ C.T 
 ````
and
````
  X @ X.T = H @ np.diag(s**2) @ H.T
````

This is because of three facts:

1. As we saw earlier in this notebook, if `X = (H*s)@C.T` is the SVD of $X$, then `X.T = (C*s)@H.T` is an SVD of $X.T$
2. If $A$ has orthonormal rows then $A @ A.T=I$ (recall $I$ is the identity matrix, a diagonal matrix with $1$ on the diagonal)
3. If $A$ has orthonormal columns then $A.T@A = I$

Let's check that `X.T@X = C @ np.diag(s**2) @ C.T`:
$$
X^T X = C \mathrm{diag}(s)H^T H \mathrm{diag}(s) C^T = C \mathrm{diag}(s^2) C.T
$$



>**Fact:** If 
$$
 X = H \mathrm{diag}(s) C^T
$$
is the SVD of $X$, then the SVD of $X^T X$ is
$$
 X^T X = C \mathrm{diag}(s^2) C^T.
$$
In particular the singular values of $X$ are the square roots of the singular values of $X^T X$.

To be honest the causality goes the other way: The vector $s^2$ is the vector of "eigenvalues" of $X^T X$, and the columns of $C$ are the "eigenvectors" of $X^T X$, and you use these to get the SVD of $X$! (We'll see this later in the course, time permitting)

The sample covariance matrix is 
````
Cov(X) = 1/(n-1) * X.T @ X,
````
and its SVD is
````
Cov(X) = (C * (1/(n-1) * s**2)  @ C.T
````
>**Fact:** If $s$ is the vector of singular values of our data set $X$, then the singular values of `Cov(X)` are (1/(n-1)) * s**2.

The singular values (also called eigenvalues) of the covariance matrix give the explained variance. Let 
````
t = (1/(n-1)) * s*s.
````
It is a vector of non-negative numbers. Let $T$ be the sum of its entries, or in Python
````
T = np.sum(t).
````

>**Fact:** The fraction of variance in the data explained by the $j$ principal component is `t[j]/ T`. The fraction of variance in the data explained by the first $d$ principal components is 
````
np.sum(t[:d]) / T.
````

Let's see this in action, by comparing our hand-crafted explanation of variance with the result from scikit-learn.

In [ ]:
## Let's do PCA on our faces data using scikit-learn as we did last week
pca = PCA(85)
transformed_data = pca.fit_transform(faces.data)

## This was the plot of explained variance
fig,ax=plt.subplots(1,1)
ax.plot(np.cumsum(pca.explained_variance_ratio_))
ax.set_xlabel('number of components')
ax.set_ylabel('cumulative explained variance');

In [ ]:
# We should be able to do the same calculation this way
# We already computed this above, but let's do it again to be safe

U,s,VT = la.svd(X,full_matrices=False)
V = VT.T

#  According to the discussion above, the singular values of the sample covariance matrix are

nobs = X.shape[0]
t = (1/(nobs-1)) * s * s

# Let T be the sum of the entries of t:

T = np.sum(t)

# The fraction of variance explained by the first d principal components is np.sum(t[:d])/T.
# The d entry of np.cumsum(t) is np.sum(t[:d])

explained_variance_ratio_SVD = np.cumsum(t) / T

fig,ax = plt.subplots(1,2,figsize=(20.,10.))

ax[0].plot(np.cumsum(pca.explained_variance_ratio_))
ax[0].set_xlabel('number of components')
ax[0].set_ylabel('cumulative explained variance')
ax[0].set_title('From scikit-learn')

ax[1].plot(explained_variance_ratio_SVD[:85])
ax[1].set_xlabel('number of components')
ax[1].set_ylabel('cumulative explained variance')
ax[1].set_title('From SVD');


## The SVD and approximating a matrix

Last week in the SVD lab and in the lecture notes on PCA, you repeatedly approximate a matrix using the SVD.


Let $X$ be a matrix with $n$ rows and $k$ columsn, and compute its trim SVD. Let $d=min(n,k)$. Compute the SVD of $X$:

````
                           H,s,CT = la.svd(X,full_matrices=False)
````

Let's write out the columns, rows, and singular values in our SVD:
$$
H = \begin{bmatrix}
| & & | \\
h_0 & \ldots  & h_{d-1} \\
| & & |
\end{bmatrix},
$$

$$
CT = \begin{bmatrix}
-- & c_0 & --\\
 & \vdots &  \\
-- & c_{d-1} & --\\
\end{bmatrix},
$$
and
$$
s=[s_0,\ldots,s_{d-1}].
$$ 


Remember that the effect of $X$ is to send $c_j$ to $s_j * h_j$ for $0\leq j < d$.  Now pick some positive integer $a$ less than $d$, and also so that $s_a>0$. Form the new matrix

````
                                                M = (H[:,:a] * s[:a]) @ CT[:a].
````
You made $M$ using the first $a$ columns of $H$, the first $a$ singular values, and the first $a$ rows of $CT$. $M$ sends $c_j$ to $s_j * h_j$ for $0\leq j< a$, and $c_j$ to zero for $a\leq j < d$. That is, $M$ drops the last few $c_j$'s, corresponding to the smallest $d-a$ singular values.  

>**Definition:** $M$ is called the "closest rank $a$ approximation" to $X.$  

For example, when you approximate the faces in our faces data base using the first $85$ singular values/principal components, what you were doing was computing the closest rank 85 approximation to your centered faces data maxtrix, and then adding back the mean. Let's do that again, using our SVD and comparing to what we found using scikit-learn.





In [ ]:
# Our centered faces data is in the matrix X. Here is its SVD

U,s,VT = la.svd(X,full_matrices=False)

# Here is the rank 85 approximation

M = (U[:,:85] * s[:85]) @ VT[:85]

# Let's compute our approximate pictures by adding back the mean

Approximate_Faces = M + faces_mean

# And then display them. 

Approximate_Faces_sklearn  = pca.inverse_transform(transformed_data)

# Plot the results
fig, ax = plt.subplots(3, 10, figsize=(15,6),
                       subplot_kw={'xticks':[], 'yticks':[]},
                       gridspec_kw=dict(hspace=0.1, wspace=0.1))

for i in range(10):
    ax[0, i].imshow(faces.images[i], cmap='binary_r')
    ax[1, i].imshow(Approximate_Faces[i].reshape(62, 47), cmap='binary_r')
    ax[2, i].imshow(Approximate_Faces_sklearn[i].reshape(62, 47), cmap='binary_r')
    
    
ax[0, 0].set_ylabel('full-dim\ninput')
ax[1, 0].set_ylabel('SVD\nreconstruction')
ax[2,0].set_ylabel('scikit\nreconstruction');